# ExpansionRx: your first molecular ML baseline

**Morgan fingerprints + random forests across nine ADMET endpoints.**

Use a **CPU runtime**. No GPU, paid account, or API key is needed. Run cells in order. Only original training data is downloaded; the final challenge test set remains untouched.

This notebook runs the versioned Python project rather than maintaining a second implementation. Results are internal scaffold-validation estimates, not official leaderboard scores.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Local override is useful for testing this notebook outside Colab.
REPO = Path(os.environ.get("EXPANSIONRX_REPO", "/content/expansionrx-baselines"))
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/antoniaval/expansionrx-baselines.git", str(REPO)], check=True)
print("Project:", REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)


## Save your experiment

Drive is recommended on Colab so you can resume after a disconnect. Authorize your own Drive when prompted. Change `SAVE_TO_DRIVE` to `False` for a temporary run, then download results before the runtime ends.

The default is 200 trees. Use a different run name if you change settings; the code rejects incompatible resume attempts.

In [ ]:
SAVE_TO_DRIVE = True
SEED = 42
TREES = 200
JOBS = 2

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path("/content/drive/MyDrive/ExpansionRx/runs")
else:
    RESULTS_ROOT = REPO / "runs"

OUTPUT = RESULTS_ROOT / f"rf_seed{SEED}_trees{TREES}"
CACHE = REPO / "data"
print("Results:", OUTPUT)


## Train and evaluate

The script verifies the dataset checksum, builds a scaffold-separated holdout, trains each endpoint on its observed labels, and compares with a training-median predictor. Completed endpoints are saved individually. Rerun this cell to resume.

Allow several minutes to tens of minutes depending on CPU allocation; the first run measures the actual time.

In [ ]:
subprocess.run([
    sys.executable, "-m", "expansionrx.baseline",
    "--output", str(OUTPUT), "--cache", str(CACHE),
    "--seed", str(SEED), "--trees", str(TREES), "--jobs", str(JOBS)
], check=True)


## Read your results

**Lower RAE is better.** Compare the forest with the training-median predictor endpoint by endpoint. Some endpoints have few observed validation labels, so treat small differences cautiously. Do not compare these numbers with the official leaderboard: the split differs, and these are point estimates rather than bootstrap-averaged scores.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

scores = pd.read_csv(OUTPUT / "metrics.csv")
display(scores)
print(json.dumps(json.loads((OUTPUT / "summary.json").read_text()), indent=2))
display(Image(filename=str(OUTPUT / "validation.png")))


## Download a lightweight results bundle

This archive includes configurations, split membership, metrics and predictions. Model files remain in the run folder to keep the download small. Never commit model binaries or personal Drive contents to GitHub.

**Next:** review label counts and errors, then run additional seeds and add the multitask model. Keep the published final test set untouched while making those decisions.

In [ ]:
import zipfile
archive = OUTPUT.parent / f"{OUTPUT.name}_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in OUTPUT.rglob("*"):
        if path.is_file() and path.suffix in {".json", ".csv", ".png"}:
            bundle.write(path, arcname=str(path.relative_to(OUTPUT)))
print("Results archive:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))
